In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

# Loading the dataset
df = pd.read_csv("Churn_Modelling.csv")
df.head()

# Dropout  so that model should not overfit
##Basic Feature engineering

## Preprocess data - Drop irrelvent data

data = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data.head()

## Geogrphy and Gender - Categorical variable
##Do Encoding for categorical variables

lable_encoder_gender = LabelEncoder()
data["Gender"] = lable_encoder_gender.fit_transform(data["Gender"])



In [52]:
## One hot encoding

from sklearn.preprocessing import OneHotEncoder

onehot_encoder_geo = OneHotEncoder()
geo_encoded_values = onehot_encoder_geo.fit_transform(data[["Geography"]])

geo_encoded_values

geo_encoded_df = pd.DataFrame(
    geo_encoded_values.toarray(),
    columns=onehot_encoder_geo.get_feature_names_out(["Geography"])
)
geo_encoded_df

# Combine One-hot encoded columns with the preprocessed data
processed_df = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)
df = processed_df.copy()
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [53]:
values=onehot_encoder_geo.get_feature_names_out(['Geography'])

In [54]:
geo_encoded_df = pd.DataFrame(
    geo_encoded_values.toarray(),
    columns=onehot_encoder_geo.get_feature_names_out(["Geography"]),
)
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [55]:
## Save encoders and scalars

with open('lable_encoder_gender.pkl', 'wb') as file:
    pickle.dump(lable_encoder_gender, file )

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

In [56]:
## Divide data into dependent and independent dataset
# Combine One_hot encoded columns with original data
combined_data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)
X = combined_data.drop("Exited", axis=1)
y = combined_data["Exited"]

## Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

#INitialize sequential network

#Hidden neurons - Dense Library

# Activation Function - Sigmoid, Tanh, relu, leaky relu
# OPtimiser - useful for back propogation - upfating the weights

#Loss Function

#Metrics [Accuracy]

# Training  Information  -> Logs in 1 folder -> Tensor board -> Visiualization



In [59]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.models import load_model
import datetime

##Build Model
X_train.shape

model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(X_train.shape[1],)),  ##HL1
        Dense(32, activation="relu"),  ##HL2
        Dense(1, activation="sigmoid"),  ##Output layer
    ]
)

model.summary()

## Compile the model
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

##Setup tensorboard
log_dir = log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

##Set up early stopping
early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

## Train the model
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback],
)

model.save('model.h5')

##Load tensorboard extention

%load_ext tensorboard
%tensorboard --logdir logs/fit

##Load the Pickle File , Trained model, Scaler pickle and onehot pickle

model=load_model('model.h5')

## Load Encoders and scaler
with open('lable_encoder_gender.pkl', 'rb') as file:
    lable_encoder_gender=pickle.load(file)

with open("onehot_encoder_geo.pkl",'rb') as file:
    onehot_encoder_geo=pickle.load(file)

with open("scaler.pkl",'rb') as file:
    scaler=pickle.load(file)


 


Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_54 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_56 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8246 - loss: 0.4025 - val_accuracy: 0.8635 - val_loss: 0.3458
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8561 - loss: 0.3540 - val_accuracy: 0.8625 - val_loss: 0.3389
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8560 - loss: 0.3480 - val_accuracy: 0.8615 - val_loss: 0.3432
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8590 - loss: 0.3445 - val_accuracy: 0.8630 - val_loss: 0.3416
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8608 - loss: 0.3409 - val_accuracy: 0.8585 - val_loss: 0.3428
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8622 - loss: 0.3391 - val_accuracy: 0.8610 - val_loss: 0.3419
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8616 - loss: 0.3362 - val_accuracy: 0.8645 - val_loss: 0.3419
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8635 - loss: 0.3348 - val_accu

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 21536), started 1:03:25 ago. (Use '!kill 21536' to kill it.)

In [58]:
## Load the Pickle file